# AI AINAK — Notebook 3: Kinematic Gesture Recognition Module

**Branch 3:** MediaPipe Pose 33-Point Skeletal Keypoint Extraction, Torso-Centered Normalization, 24-Frame Temporal Buffer, and 2-Layer GestureLSTM Inference.

---

## 1. Dataset Taxonomy & Asset Specification

This notebook details every dataset connected to the kinematic gesture recognition branch:

| Category | Dataset Name | Source / URL | License | Purpose in AI AINAK |
| :--- | :--- | :--- | :--- | :--- |
| **Training Dataset (Keypoint Extractor)** | **Google BlazePose 3D Dataset** | [MediaPipe Models](https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task) | Apache-2.0 | Pre-trained MediaPipe Tasks Pose Landmarker Lite model predicting 33 3D bodily keypoints with visibility scores. Auto-downloaded below. |
| **Training Dataset (Temporal LSTM)** | **AI AINAK Multi-Subject Kinematic Dataset (Subjects p01, p02, p04)** | `datasets/gesture-sessions/gesture_manifest.csv` | Project License | 96 normalized kinematic sequences (24 frames × 132 features) recorded across 3 independent participants. Classes: `pointing`, `waving`, `stop`, `neutral`. |
| **Evaluation Dataset** | **AI AINAK Subject `subject_p03` Held-Out Split** | `datasets/gesture-sessions/` | Project License | 32 kinematic sequences from a completely independent participant. GestureLSTM achieves **100.0% accuracy, 1.0000 Macro-F1, and zero cross-class confusion** (reported in `artifacts/gesture/gesture_lstm.metrics.json`). *Inspectable via optional snippet below.* |
| **Demonstration Input** | **Paired Session `session-multimodal-01`** | Project Repository / Google Drive | Project License | Real-world corridor video recording (`video.mp4`, 2.30 MB, 100 frames @ 30.0 FPS, 3.33s duration). Synchronized with paired audio starting at $t=0.0\,	ext{s}$ per `recording.json`. |

---

## 2. Checkpoint & Demonstration Media Manifest

| Asset Name | Target Path | File Size | Origin / Acquisition Method |
| :--- | :--- | :--- | :--- |
| **MediaPipe Pose Landmarker** | `checkpoints/pose_landmarker_lite.task` | ~5.78 MB | Public official task model (Google Cloud Storage). Auto-downloaded below if missing. |
| **GestureLSTM Checkpoint** | `checkpoints/gesture_lstm.pt` | ~1.07 MB | Trained AI AINAK project checkpoint (2-layer unidirectional LSTM, hidden size 128). |
| **Demonstration Video** | `datasets/recordings/session-multimodal-01/video.mp4` | ~2.30 MB | Synchronized session recording video. |
| **Session Metadata** | `datasets/recordings/session-multimodal-01/recording.json` | ~837 B | Verified session clock and pairing assertions. |


In [ ]:
# Step 1: Install Dependencies
!pip install -q mediapipe opencv-python torch numpy matplotlib


In [ ]:
# Environment Detection, Google Drive Setup & Path Configuration
import os, sys, shutil, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("=" * 70)
    print("  AI AINAK: Google Colab Runtime Environment Detected")
    print("=" * 70)
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        print("  [OK] Google Drive mounted successfully at /content/drive")
    except Exception as e:
        print(f"  [NOTICE] Drive mount skipped or already mounted: {e}")

    # Standard Google Drive project folder
    DRIVE_AINAK = Path("/content/drive/MyDrive/AINAK_Colab")
    
    # Candidate project locations in priority order
    drive_candidates = [
        DRIVE_AINAK,
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), DRIVE_AINAK)
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    print("=" * 70)
    print("  AI AINAK: Local Runtime Environment Detected")
    print("=" * 70)
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"  Project Root resolved to: {PROJECT_ROOT}")

# Create standard subdirectories if not present
(PROJECT_ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "datasets" / "recordings" / "session-multimodal-01").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR = PROJECT_ROOT / "outputs"



In [ ]:
# Checkpoint & Dataset Acquisition Utilities
def download_url(url: str, dest_path: Path, desc: str = "") -> Path:
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.is_file() and dest_path.stat().st_size > 0:
        print(f"  [FOUND] {desc or dest_path.name} already present ({dest_path.stat().st_size / 1e6:.2f} MB). Skipping download.")
        return dest_path
    print(f"  [DOWNLOADING] {desc or dest_path.name} from:
    {url}")
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (AINAK-Colab-Environment)"})
        with urllib.request.urlopen(req) as resp, open(tmp_path, "wb") as f:
            shutil.copyfileobj(resp, f)
        tmp_path.replace(dest_path)
        print(f"  [DOWNLOAD COMPLETE] {dest_path.name} ({dest_path.stat().st_size / 1e6:.2f} MB)")
        return dest_path
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        raise RuntimeError(f"Failed to download {dest_path.name} from {url}: {e}")

def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return candidates[2] # Return standard checkpoints/ location as target

def find_session_file(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return candidates[0] # Return standard datasets/recordings/<session>/ location

def unpack_starter_bundle_if_available():
    bundle_candidates = [
        PROJECT_ROOT / "ainak_colab_starter_bundle.zip",
        PROJECT_ROOT / "outputs" / "ainak_colab_starter_bundle.zip",
        Path("/content/ainak_colab_starter_bundle.zip"),
    ]
    for b in bundle_candidates:
        if b.is_file() and b.stat().st_size > 0:
            print(f"  [UNPACKING] Found starter bundle at {b}. Extracting to {PROJECT_ROOT}...")
            import zipfile
            with zipfile.ZipFile(str(b), "r") as zf:
                zf.extractall(str(PROJECT_ROOT))
            print("  [UNPACK COMPLETE] Starter checkpoints and demonstration media extracted.")
            return True
    return False

def run_preflight_audit(required_items):
    print("=" * 80)
    print("  AI AINAK PRE-FLIGHT ASSET & DATASET INTEGRITY AUDIT")
    print("=" * 80)
    all_ok = True
    for item in required_items:
        path = Path(item["path"])
        min_size = item.get("min_size", 1)
        role = item.get("role", "Asset")
        desc = item.get("desc", path.name)
        
        if path.is_file() and path.stat().st_size >= min_size:
            size_mb = path.stat().st_size / 1e6
            status = "PASSED"
            print(f"  [OK] {status:<6} | {path.name:<28} | {size_mb:6.2f} MB | {role:<16} | {desc}")
        else:
            all_ok = False
            status = "FAILED"
            print(f"  [!!] {status:<6} | {path.name:<28} | MISSING   | {role:<16} | {desc}")
            print(f"       -> Expected location: {path}")
            if "download_url" in item:
                print(f"       -> Download URL: {item['download_url']}")
            elif "action" in item:
                print(f"       -> Action required: {item['action']}")
    print("=" * 80)
    if not all_ok:
        raise FileNotFoundError(
            "Pre-flight audit failed: One or more required datasets or model checkpoints are missing. "
            "Please review the missing items above and obtain them before proceeding."
        )
    print("  All required assets verified. Proceeding to inference...
")



## Step 2: Automated Dataset & Checkpoint Acquisition
Here we acquire the public MediaPipe Pose Landmarker task bundle from Google Cloud Storage automatically and resolve the GestureLSTM checkpoint and session video.


In [ ]:
# Check for starter bundle first
unpack_starter_bundle_if_available()

# 1. Acquire MediaPipe Pose Landmarker Task Bundle (Official Google Storage URL)
POSE_TASK_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
pose_task_path = find_checkpoint("pose_landmarker_lite.task", "gesture")
if not pose_task_path.is_file():
    pose_target = PROJECT_ROOT / "checkpoints" / "pose_landmarker_lite.task"
    pose_task_path = download_url(POSE_TASK_URL, pose_target, "MediaPipe Pose Landmarker Task")

# 2. Resolve GestureLSTM Checkpoint & Session Video
lstm_path = find_checkpoint("gesture_lstm.pt", "gesture")
video_path = find_session_file("session-multimodal-01", "video.mp4")
metadata_path = find_session_file("session-multimodal-01", "recording.json")

print(f"Pose Landmarker Path: {pose_task_path}")
print(f"GestureLSTM Checkpoint Path: {lstm_path}")
print(f"Session Video Path: {video_path}")


### Optional: Gesture Training Dataset Inspection
Set `INSPECT_GESTURE_DATASET = True` to inspect the training and evaluation participant splits in `datasets/gesture-sessions/`.


In [ ]:
INSPECT_GESTURE_DATASET = False

if INSPECT_GESTURE_DATASET:
    manifest_csv = PROJECT_ROOT / "datasets" / "gesture-sessions" / "gesture_manifest.csv"
    if manifest_csv.is_file():
        import pandas as pd
        df = pd.read_csv(manifest_csv)
        print(f"[OK] Loaded Gesture Manifest: {len(df)} total sequences across {df['subject_id'].nunique()} participants.")
        print("Sequence count by class and subject:")
        print(pd.crosstab(df['subject_id'], df['label']))
    else:
        print(f"[NOTICE] Gesture manifest not found at: {manifest_csv}")
else:
    print("[INFO] Skipping gesture training manifest inspection (demonstration mode active).")


## Step 3: Pre-Flight Asset Integrity & MediaPipe Validation
We verify that all required gesture models and video files exist and validate MediaPipe initialization.


In [ ]:
# Execute pre-flight integrity audit
required_manifest = [
    {
        "path": pose_task_path,
        "min_size": 5_000_000,
        "role": "Pose Landmarker",
        "desc": "MediaPipe Tasks 33-point pose landmarker",
        "download_url": POSE_TASK_URL
    },
    {
        "path": lstm_path,
        "min_size": 1_000_000,
        "role": "Sequence Classifier",
        "desc": "Trained 2-layer GestureLSTM (4 classes)",
        "action": "Upload checkpoints/gesture_lstm.pt to Google Drive or project folder"
    },
    {
        "path": video_path,
        "min_size": 2_000_000,
        "role": "Demonstration Input",
        "desc": "Paired session video (session-multimodal-01)",
        "action": "Upload session-multimodal-01/video.mp4 to Google Drive or project folder"
    },
    {
        "path": metadata_path,
        "min_size": 100,
        "role": "Clock Metadata",
        "desc": "Session clock & pairing assertion",
        "action": "Ensure recording.json is present in session-multimodal-01"
    }
]
run_preflight_audit(required_manifest)


## Step 4: MediaPipe Pose Tasks API & Normalization Implementation
Defines landmark normalization and 2-layer GestureLSTM architecture matching `ainak.models.gesture`.


In [ ]:
import mediapipe as mp
import cv2
import torch
from torch import nn
from collections import deque
import numpy as np
import time

GESTURE_LABELS = ("pointing", "waving", "stop", "neutral")

# 1. Landmark Normalizer (33 keypoints -> 132-D vector)
def normalize_landmarks(landmarks_33x4: np.ndarray) -> np.ndarray:
    p = np.asarray(landmarks_33x4, dtype=np.float32).reshape(33, 4).copy()
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    # MediaPipe landmarks: Left shoulder=11, Right shoulder=12, Left hip=23, Right hip=24
    center = (p[23, :3] + p[24, :3]) / 2.0
    scale = max(float(np.linalg.norm(p[11, :2] - p[12, :2])), 1e-3)
    p[:, :3] = (p[:, :3] - center) / scale
    p[p[:, 3] < 0.2, :3] = 0.0
    p[:, :3] = np.clip(p[:, :3], -5.0, 5.0)
    p[:, 3] = np.clip(p[:, 3], 0.0, 1.0)
    return p.reshape(-1)

# 2. GestureLSTM Architecture (2-layer unidirectional)
class GestureLSTM(nn.Module):
    def __init__(self, feature_dim: int = 132, hidden_size: int = 128, classes: int = 4, dropout: float = 0.2, layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(feature_dim, hidden_size, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden_size)
        self.classifier = nn.Linear(hidden_size, classes)

    def forward(self, x: torch.Tensor):
        _, (h, _) = self.lstm(x)
        embedding = self.norm(h[-1])
        return self.classifier(embedding), embedding

# Load GestureLSTM model
lstm_payload = torch.load(str(lstm_path), map_location="cpu", weights_only=False)
gesture_model = GestureLSTM(feature_dim=132, hidden_size=128, classes=4, layers=2)
gesture_model.load_state_dict(lstm_payload["state_dict"])
gesture_model.eval()

# 3. MediaPipe PoseLandmarker Tasks API Wrapper
base_options = mp.tasks.BaseOptions(model_asset_path=str(pose_task_path))
options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
landmarker = mp.tasks.vision.PoseLandmarker.create_from_options(options)

print("PoseLandmarker Tasks API & GestureLSTM initialized successfully.")


## Step 5: Process Video Frames, Extract Skeletons & Run GestureLSTM
We feed video frames to MediaPipe, buffer 24 frames, and run GestureLSTM forward passes.


In [ ]:
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)

buffer = deque(maxlen=24)
timestamps = []
features_128d = []
predicted_classes = []
confidences = []
landmarks_history = []
latencies_ms = []
sample_skeleton_frames = []

frame_idx = 0
print("Extracting pose landmarks and running GestureLSTM...")
t0 = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    ts = frame_idx / fps
    frame_ms = int(ts * 1000)
    frame_idx += 1

    t_start = time.perf_counter()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect_for_video(mp_image, frame_ms)

    if result.pose_landmarks and len(result.pose_landmarks) > 0:
        lm_list = result.pose_landmarks[0]
        points = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in lm_list], dtype=np.float32)
        norm_132 = normalize_landmarks(points)
        buffer.append(norm_132)

        if len(buffer) == 24:
            seq_array = np.stack(buffer)[None, :]
            with torch.no_grad():
                x_tensor = torch.from_numpy(seq_array)
                logits, emb_128 = gesture_model(x_tensor)
                probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
            
            top_idx = int(probs.argmax())
            top_cls = GESTURE_LABELS[top_idx]
            top_conf = float(probs[top_idx])
            emb_vector = emb_128[0].cpu().numpy()
        else:
            top_cls = "buffering"
            top_conf = 0.0
            emb_vector = np.zeros(128, dtype=np.float32)

        landmarks_history.append(points)
    else:
        norm_132 = np.zeros(132, dtype=np.float32)
        emb_vector = np.zeros(128, dtype=np.float32)
        top_cls = "neutral"
        top_conf = 0.0
        landmarks_history.append(np.zeros((33, 4), dtype=np.float32))

    t_latency = (time.perf_counter() - t_start) * 1000

    timestamps.append(ts)
    features_128d.append(emb_vector)
    predicted_classes.append(top_cls)
    confidences.append(top_conf)
    latencies_ms.append(t_latency)

    if frame_idx % 25 == 0 or frame_idx == 1:
        sample_skeleton_frames.append((ts, rgb.copy(), landmarks_history[-1], top_cls, top_conf))

cap.release()
print(f"Completed gesture extraction: {len(timestamps)} frames processed in {time.time() - t0:.2f}s.")
print(f"Mean Gesture Latency: {np.mean(latencies_ms):.2f} ms | p95 Latency: {np.percentile(latencies_ms, 95):.2f} ms")


## Step 6: Skeletal Overlay Visualization


In [ ]:
import matplotlib.pyplot as plt

POSE_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (24, 26), (25, 27), (26, 28),
    (27, 29), (28, 30), (29, 31), (30, 32),
    (0, 11), (0, 12)
]

fig, axes = plt.subplots(1, min(3, len(sample_skeleton_frames)), figsize=(16, 5))
if not isinstance(axes, np.ndarray):
    axes = [axes]

for idx, (ts, rgb_frame, lms, cls_name, conf) in enumerate(sample_skeleton_frames[:3]):
    h, w = rgb_frame.shape[:2]
    annotated = rgb_frame.copy()

    if lms.any():
        for i, j in POSE_CONNECTIONS:
            p1, p2 = lms[i], lms[j]
            if p1[3] > 0.3 and p2[3] > 0.3:
                cv2.line(annotated, (int(p1[0]*w), int(p1[1]*h)), (int(p2[0]*w), int(p2[1]*h)), (16, 185, 129), 3)
        for pt in lms:
            if pt[3] > 0.3:
                cv2.circle(annotated, (int(pt[0]*w), int(pt[1]*h)), 4, (245, 158, 11), -1)

    axes[idx].imshow(annotated)
    axes[idx].set_title(f"Time: {ts:.2f}s | Gesture: {cls_name.upper()} ({conf*100:.0f}%)", fontsize=11, fontweight="bold")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()


## Step 7: Export Gesture Outputs to `gesture_outputs.npz`


In [ ]:
gesture_out_path = OUTPUTS_DIR / "gesture_outputs.npz"

np.savez_compressed(
    str(gesture_out_path),
    timestamps=np.array(timestamps, dtype=np.float64),
    features=np.array(features_128d, dtype=np.float32),
    landmarks=np.array(landmarks_history, dtype=np.float32),
    predicted_classes=np.array(predicted_classes),
    confidences=np.array(confidences, dtype=np.float32),
    latencies_ms=np.array(latencies_ms, dtype=np.float32)
)

print(f"Successfully exported gesture outputs to: {gesture_out_path}")
print(f"File size: {gesture_out_path.stat().st_size / 1e3:.1f} KB")
print(f"Feature matrix shape: {np.array(features_128d).shape} (Matches 128-D Transformer contract)")
